In [6]:
%cd ~/Desktop

/Users/omarreyald/Desktop


/Users/omarreyald/Library/Python/3.12/lib/python/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/Users/omarreyald/Library/Python/3.12/lib/python/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [9]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px

# =====================================================================
# 1. MOTOR DE DATOS (data_engine)
# =====================================================================
def download_portfolio_data(tickers, start_date, end_date):
    data = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        progress=False,
        auto_adjust=True,
    )
    if data.empty:
        raise ValueError("yfinance devolvió un DataFrame vacío.")
    
    if isinstance(data.columns, pd.MultiIndex):
        if "Close" in data.columns.get_level_values(0):
            data = data["Close"]
        else:
            raise KeyError("Precio 'Close' no encontrado.")
    elif "Close" in data.columns:
        data = data["Close"]

    data = data.ffill().dropna()
    log_returns = np.log(data / data.shift(1)).dropna()
    return data, log_returns

def calculate_portfolio_stats(log_returns, trading_days=252):
    expected_returns = log_returns.mean() * trading_days
    cov_matrix = log_returns.cov() * trading_days
    corr_matrix = log_returns.corr()
    return expected_returns, cov_matrix, corr_matrix

# =====================================================================
# 2. MOTOR ESTOCÁSTICO Y RIESGO (montecarlo_gbm)
# =====================================================================
def simulate_gbm(S0, mu, sigma, T_years, n_sims, trading_days=252):
    dt = 1 / trading_days 
    steps = int(T_years * trading_days)
    
    price_paths = np.zeros((steps + 1, n_sims))
    price_paths[0] = S0
    
    Z = np.random.standard_normal((steps, n_sims))
    growth_factor = np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    
    for t in range(1, steps + 1):
        price_paths[t] = price_paths[t-1] * growth_factor[t-1]
        
    return price_paths

def calculate_risk_metrics(S0, final_values, confidence_level=0.99):
    pnl = final_values - S0
    alpha = 1.0 - confidence_level
    var = -np.percentile(pnl, alpha * 100)
    worst_losses = pnl[pnl <= -var]
    es = -np.mean(worst_losses)
    return var, es

# =====================================================================
# 3. INTERFAZ VISUAL (Streamlit App)
# =====================================================================
st.set_page_config(page_title="Risk Engine - Portafolios", layout="wide")

st.title("📊 Motor de Valuación y Riesgo Estocástico")
st.markdown("Plataforma interactiva para proyección de portafolios mediante Movimiento Browniano Geométrico y cálculo de métricas de riesgo ($VaR$ y $ES$).")

st.sidebar.header("Parámetros del Portafolio")
tickers_input = st.sidebar.text_input("Tickers (separados por coma)", "SPY, QQQ, TLT, GLD")
start_date = st.sidebar.date_input("Fecha de inicio histórico", value=pd.to_datetime("2020-01-01"))
end_date = st.sidebar.date_input("Fecha de fin histórico", value=pd.to_datetime("2026-01-01"))

inversion_inicial = st.sidebar.number_input("Inversión Inicial (MXN)", value=1000000, step=100000)
horizonte = st.sidebar.slider("Horizonte de Proyección (Años)", 1, 5, 1)
simulaciones = st.sidebar.selectbox("Número de Simulaciones (MC)", [1000, 5000, 10000])

if st.sidebar.button("Calcular y Simular Riesgo"):
    with st.spinner("Procesando pipeline cuantitativo..."):
        # 1. Extracción de Datos
        tickers = [t.strip() for t in tickers_input.split(',')]
        _, retornos = download_portfolio_data(tickers, start_date, end_date)
        rendimientos_esp, matriz_cov, _ = calculate_portfolio_stats(retornos)
        
        # Asumimos un portafolio equiponderado para esta demo
        pesos = np.ones(len(tickers)) / len(tickers)
        retorno_port = np.sum(pesos * rendimientos_esp)
        vol_port = np.sqrt(np.dot(pesos.T, np.dot(matriz_cov, pesos)))
        
        # 2. Simulación Monte Carlo
        trayectorias = simulate_gbm(inversion_inicial, retorno_port, vol_port, horizonte, simulaciones)
        valores_finales = trayectorias[-1, :]
        
        # 3. Métricas de Riesgo
        var_99, es_99 = calculate_risk_metrics(inversion_inicial, valores_finales, 0.99)
        
        # 4. Visualización
        col1, col2, col3 = st.columns(3)
        col1.metric("Retorno Esperado Anual", f"{retorno_port*100:.2f}%")
        col2.metric("Valor en Riesgo (VaR 99%)", f"-${var_99:,.2f}")
        col3.metric("Expected Shortfall (ES 99%)", f"-${es_99:,.2f}")
        
        st.subheader("Trayectorias de Precio Simuladas (Submuestra de 100 caminos)")
        fig = px.line(trayectorias[:, :100])
        fig.update_layout(xaxis_title="Días", yaxis_title="Valor del Portafolio", showlegend=False)
        st.plotly_chart(fig, use_container_width=True)

        st.subheader("Distribución del Valor Final")
        fig_dist = px.histogram(valores_finales, nbins=50, histnorm='probability density')
        fig_dist.add_vline(x=inversion_inicial - var_99, line_dash="dash", line_color="red", annotation_text="VaR 99%")
        st.plotly_chart(fig_dist, use_container_width=True)

Overwriting app.py


In [11]:
!streamlit run app.py

2026-06-02 00:51:24.881 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.1.78:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
2026-06-02 00:52:11.820 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-02 00:52:11.860 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-02 00:52:50.916 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `us